In [50]:
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
from ezc3d import c3d
from scipy.signal import butter, filtfilt

In [51]:
import numpy as np
from scipy.signal import butter, filtfilt

def lowpass_filter_data(data, cutoff=4, fs=100, order=4):
    """
    Apply a Butterworth lowpass filter to the data along each column.

    Parameters:
      data   : 2D NumPy array of shape (n, 5)
      cutoff : Cutoff frequency in Hz (default is 4 Hz)
      fs     : Original sampling frequency (default is 100 Hz)
      order  : Order of the Butterworth filter (default is 4)
    
    Returns:
      Filtered data as a NumPy array with the same shape as the input.
    """
    nyq = 0.5 * fs                  # Nyquist Frequency
    normal_cutoff = cutoff / nyq    # Normalized cutoff frequency
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    # Use filtfilt to avoid phase shift.
    filtered_data = filtfilt(b, a, data, axis=0)
    return filtered_data

def downsample(data, orig_rate=100, new_rate=10):
    """
    Downsample the data from orig_rate to new_rate by taking every (orig_rate/new_rate)-th sample.
    
    Parameters:
      data      : 2D NumPy array of shape (n, 5)
      orig_rate : Original sampling frequency (Hz)
      new_rate  : Desired sampling frequency (Hz)
    
    Returns:
      Downsampled data.
    """
    factor = orig_rate // new_rate
    return data[::factor]

def find_near_zero_blocks(data, tol=1e-2, min_gap=30):
    """
    Identify the start indices of contiguous blocks where all 5 columns are near zero.
    A new near-zero block is only accepted if it is at least 'min_gap' samples after the previous one.

    Parameters:
      data    : 2D NumPy array of shape (n, 5)
      tol     : Tolerance to consider a value as zero (default is 1e-2)
      min_gap : Minimum number of samples between consecutive near-zero blocks (default is 30)
    
    Returns:
      A NumPy array of filtered start indices for near-zero blocks.
    """
    # Create a boolean mask that is True when all 5 columns are nearly zero.
    near_zero = np.all(np.abs(data[:,:4]) < tol, axis=0)
    
    # Compute differences to identify transitions from non-zero to near-zero.
    diff = np.diff(near_zero.astype(int))
    starts = np.where(near_zero == True)[0]   # +1 to point to the first True value in the block
    
    # If the very first sample is near zero, include index 0.
    if near_zero[0]:
        starts = np.insert(starts, 0, 0)
    
    # Filter out indices that are too close together.
    filtered_starts = []
    if len(starts) > 0:
        filtered_starts.append(starts[0])
        for idx in starts[1:]:
            if idx - filtered_starts[-1] >= min_gap:
                filtered_starts.append(idx)
    
    return np.array(filtered_starts)

def find_cycle_boundaries(data, tol=1e-2, min_gap=30):
    """
    Identify cycle boundaries in the periodic data. Here, a cycle is assumed to start at the first 
    near-zero block and end at the start of the next near-zero block that is at least 'min_gap' samples later.
    
    Parameters:
      data    : 2D NumPy array (downsampled) of shape (n, 5)
      tol     : Tolerance to consider a value as zero (default is 1e-2)
      min_gap : Minimum number of samples between consecutive near-zero blocks (default is 30)
    
    Returns:
      A tuple (cycle_start, cycle_stop) indicating the start and stop indices of one cycle.
    """
    near_zero_starts = find_near_zero_blocks(data, tol, min_gap)
    
    if len(near_zero_starts) < 2:
        raise ValueError("Not enough near-zero regions found to determine a cycle!")
    
    cycle_start = near_zero_starts[0]
    cycle_stop = near_zero_starts[1]
    return cycle_start, cycle_stop


In [52]:
# def lowpass_filter(data, cutoff=5, fs=100, order=4):
#     nyquist = 0.5 * fs
#     normal_cutoff = cutoff / nyquist
#     b, a = butter(order, normal_cutoff, btype='low', analog=False)
#     y = filtfilt(b, a, data, axis=0)
#     return y

def extract_gait_cycle(c, gait_duration=3,mode='fft'):

    times = c["parameters"]["EVENT"]["TIMES"]['value']
    contexts = c["parameters"]["EVENT"]["CONTEXTS"]['value']
    labels = c["parameters"]["EVENT"]["LABELS"]['value']
    gait_points = []

    for i in range(len(labels)):
        if (labels[i] == "Foot Strike1") and (contexts[i] == "Right"):
            gait_start = int(times[1,i]*100)
        if (labels[i] == "Foot Strike2") and (contexts[i] == "Right"):
            gait_end = int(times[1,i]*100+1)

    trial_data = c['data']['points']
    labels = c['parameters']['POINT']['LABELS']['value']

    #[15,4,16,5,20,9]
    r_ftc = labels.index("R_FTC") if "R_FTC" in labels else RuntimeError("R_FTC not found")
    l_ftc = labels.index("L_FTC") if "L_FTC" in labels else RuntimeError("L_FTC not found")
    r_fle = labels.index("R_FLE") if "R_FLE" in labels else RuntimeError("R_FLE not found")
    l_fle = labels.index("L_FLE") if "L_FLE" in labels else RuntimeError("L_FLE not found")
    r_fal = labels.index("R_FAL") if "R_FAL" in labels else RuntimeError("R_FAL not found")
    l_fal = labels.index("L_FAL") if "L_FAL" in labels else RuntimeError("L_FAL not found")
    r_fcc = labels.index("R_FCC") if "R_FCC" in labels else RuntimeError("R_FCC not found")
    r_fm1 = labels.index("R_FM1") if "R_FM1" in labels else RuntimeError("R_FM1 not found")
    l_fcc = labels.index("L_FCC") if "L_FCC" in labels else RuntimeError("L_FCC not found")
    l_fm1 = labels.index("L_FM1") if "L_FM1" in labels else RuntimeError("L_FM1 not found")

    sjn = labels.index("SJN") if "SJN" in labels else RuntimeError("SJN not found")
    sxs = labels.index("SXS") if "SXS" in labels else RuntimeError("SXS not found")

    r_leglength_ind = c['parameters']['SUBJECT']['LABELS']['value'].index("R_legLength") if "R_legLength" in c['parameters']['SUBJECT']['LABELS']['value'] else RuntimeError("R_LEGLENGTH not found")
    l_leglength_ind = c['parameters']['SUBJECT']['LABELS']['value'].index("L_legLength") if "L_legLength" in c['parameters']['SUBJECT']['LABELS']['value'] else RuntimeError("L_LEGLENGTH not found")
    weight = c['parameters']['SUBJECT']['LABELS']['value'].index("weight") if "weight" in c['parameters']['SUBJECT']['LABELS']['value'] else RuntimeError("Weight not found")
    height = c['parameters']['SUBJECT']['LABELS']['value'].index("height") if "height" in c['parameters']['SUBJECT']['LABELS']['value'] else RuntimeError("Height not found")
    r_leglength = c['parameters']['SUBJECT']['VALUES']['value'][r_leglength_ind]
    l_leglength = c['parameters']['SUBJECT']['VALUES']['value'][l_leglength_ind]
    weight = c['parameters']['SUBJECT']['VALUES']['value'][weight]
    height = c['parameters']['SUBJECT']['VALUES']['value'][height]

    muscle_index = [r_ftc,l_ftc,r_fle,l_fle,r_fal,l_fal,sxs,sjn,r_fcc,r_fm1,l_fcc,l_fm1]
    joint_data = trial_data[:,muscle_index,:]
    joint_states = np.zeros((joint_data.shape[2],4))

    data_len = joint_data.shape[2]

    first_pos = trial_data[0,sxs,0]
    last_pos = trial_data[0,sxs,-1]

    if last_pos > first_pos:
        forward_walk = True
    else:
        forward_walk = False

    speed_muscle = trial_data[:,sxs,:]
    speed = (np.abs(speed_muscle[0,-1]-speed_muscle[0,0])/(len(speed_muscle[0,:])))/10
    #data_point[:,-1] = speed

    if forward_walk:
        for i in range(data_len):
            joint_states[i,0] = np.arctan2((joint_data[0,2,i] - joint_data[0,0,i])   , (-joint_data[2,2,i] + joint_data[2,0,i])) 
            joint_states[i,1] = np.arctan2((joint_data[0,4,i] - joint_data[0,2,i])   , (-joint_data[2,4,i] + joint_data[2,2,i]))  - joint_states[i,0]
            joint_states[i,2] = np.arctan2((joint_data[0,3,i] - joint_data[0,1,i])   , (-joint_data[2,3,i] + joint_data[2,1,i]))
            joint_states[i,3] = np.arctan2((joint_data[0,5,i] - joint_data[0,3,i])   , (-joint_data[2,5,i] + joint_data[2,3,i]))  - joint_states[i,3]


    else:
        for i in range(data_len):
            joint_states[i,0] = np.arctan2((-joint_data[0,2,i] + joint_data[0,0,i]) , (-joint_data[2,2,i] + joint_data[2,0,i])) 
            joint_states[i,1] = np.arctan2((-joint_data[0,4,i] + joint_data[0,2,i]) , (-joint_data[2,4,i] + joint_data[2,2,i]))     - joint_states[i,0]
            joint_states[i,2] = np.arctan2((-joint_data[0,3,i] + joint_data[0,1,i]) , (-joint_data[2,3,i] + joint_data[2,1,i]))
            joint_states[i,3] = np.arctan2((-joint_data[0,5,i] + joint_data[0,3,i]) , (-joint_data[2,5,i] + joint_data[2,3,i]))     - joint_states[i,3]


    total_gait_sample = 32
    joint_states = lowpass_filter_data(joint_states)
    # gait_start, gait_end = find_cycle_boundaries(joint_states, tol=1e-1, min_gap=30)
    if mode == 'fft':
        joint_states = joint_states[gait_start:gait_end,:]
        joint_states = downsample(joint_states, orig_rate=100, new_rate=10)
    
    if mode == 'time':
        return joint_states
    else:
    
        if joint_states.shape[0] > total_gait_sample:
            print(f'gait cycle: {joint_states.shape[0]} longer than {total_gait_sample} seconds for speed: {speed}')
            return None,None,None,None,None
        else:      
            period = joint_states.shape[0] / total_gait_sample
        while joint_states.shape[0] < total_gait_sample:
            joint_states = np.concatenate((joint_states, joint_states),axis=0)  # Stack horizontally
        joint_states = joint_states[:total_gait_sample,:]
        
        joint_states_rfft = np.fft.rfft(joint_states, n=32,axis=0)
        real_fft = np.real(joint_states_rfft)
        imag_fft = np.imag(joint_states_rfft)
        joint_states_rfft = np.stack((real_fft, imag_fft), axis=2)
        freq_values = np.fft.rfftfreq(32, 1/10)
        encoder_vec = np.empty((3))   # init_pos + speed + r_leglength + l_leglength + ramp_angle = 0
        # encoder_vec[0:4] = joint_states[:,0]
        encoder_vec[0] = speed/2.4
        encoder_vec[1] = r_leglength /1.0
        encoder_vec[2] = l_leglength /1.0
        # encoder_vec[3] = weight / 100  # 100 is the maximum weight in the dataset
        # encoder_vec[4] = height / 2    # 1.91 m is the maximum height in the dataset

        encoder_vec = encoder_vec[np.newaxis, :]
        joint_states_rfft = joint_states_rfft[np.newaxis, :, :] #179 is the maximum value in the dataset

        return joint_states_rfft, freq_values, encoder_vec, joint_states, period

**Loop over Folders to create Dataset**

In [53]:
import os
gait_duration = 3.2
folders = [d for d in os.listdir(os.path.join(os.getcwd(),'dataset')) if os.path.isdir(os.path.join(os.path.join(os.getcwd(),'dataset'), d))]
k=0
if os.path.exists(f"gait reference fft_plots_") == False:
    os.mkdir(f"gait reference fft_plots_")

for folder in folders:
    folder_len = len(folder)
    files = [f for f in os.listdir(os.path.join(os.getcwd(),'dataset',folder)) if f.endswith('.c3d')]
    
    # print(f"{k}/{len(folders)} is processed current folder is {folder}") 
    i = 0
    for file in files:
        if 'ST' not in file:
            c = c3d(os.path.join('dataset',folder,file))
            joint_states_rfft, freq_values, encoder_vec, joint_states, period = extract_gait_cycle(c,gait_duration=gait_duration)
            if joint_states_rfft is not None:
                if i == 0:
                    folder_output_state = joint_states_rfft
                    folder_input_vector = encoder_vec
                    period_data = period
                else:
                    folder_output_state = np.vstack((folder_output_state,joint_states_rfft))
                    folder_input_vector = np.vstack((folder_input_vector,encoder_vec))
                    period_data = np.vstack((period_data,period))
                i+=1
    # print('folder_output_state_shape: ',folder_output_state.shape)
    # print('folder input vector shape: ',folder_input_vector.shape)
    # print('-----------------------------------')
    if k == 0:
        total_output_state = folder_output_state
        total_input_vector = folder_input_vector
        total_period = period_data
    else:
        total_output_state = np.vstack((total_output_state,folder_output_state))
        total_input_vector = np.vstack((total_input_vector,folder_input_vector))
        total_period = np.vstack((total_period,period_data))
    k+=1
#     np.save(f"ref_gait_library_duration{gait_duration}/{folder}_output_state.npy",folder_output_state)
#     np.save(f"ref_gait_library_duration{gait_duration}/{folder}_input_vector.npy",folder_input_vector)

# if os.path.exists(f"gait reference fft{freq_values[-1]:.2f}") == False:
#     os.mkdir(f"gait reference fft{freq_values[-1]:.2f}")

np.save(f"gait reference phase 2/output_fft_constants.npy",total_output_state)
np.save(f"gait reference phase 2/input_vector.npy",total_input_vector)
np.save(f"gait reference phase 2/period.npy",total_period)

print('total output state ',total_output_state.shape)
print('total input vector ',total_input_vector.shape)
print(f"period shape: {np.array(total_period).shape}")

gait cycle: 37 longer than 32 seconds for speed: 0.27025162842655015
gait cycle: 35 longer than 32 seconds for speed: 0.2593959968163885
gait cycle: 34 longer than 32 seconds for speed: 0.21001275549129567
gait cycle: 34 longer than 32 seconds for speed: 0.20499115922543282
gait cycle: 33 longer than 32 seconds for speed: 0.20587283881728302
gait cycle: 33 longer than 32 seconds for speed: 0.3614404296875
gait cycle: 33 longer than 32 seconds for speed: 0.33684120025021963
gait cycle: 33 longer than 32 seconds for speed: 0.30715705023871526
gait cycle: 34 longer than 32 seconds for speed: 0.25347105006167764
gait cycle: 34 longer than 32 seconds for speed: 0.23401031494140626
gait cycle: 33 longer than 32 seconds for speed: 0.23715816353133579
gait cycle: 33 longer than 32 seconds for speed: 0.2619328633927392
gait cycle: 35 longer than 32 seconds for speed: 0.34599531457779253
gait cycle: 33 longer than 32 seconds for speed: 0.31201024715137105
gait cycle: 38 longer than 32 seconds fo

**Normalization, Flatten and Recover Shape Functions**

In [54]:
def normalize_and_flatten(output_state, period_data):
    """
    Args:
        output_state: Numpy array of shape (N, 17, 4, 2)
                      (N samples, 17 Freqs, 4 Joints, 2 Real/Imag)
        period_data:  Numpy array of shape (N, 1)
    Returns:
        normalized_targets: (N, 137)
        mean: (136,) - Saved for denormalization
        std:  scalar - Saved for denormalization
    """
    # 1. FLATTEN: We flatten dimensions (1, 2, 3) into a single vector.
    # We use order='C' (default), which reads indices row-by-row.
    # Order: Freq 0 (all joints) -> Freq 1 (all joints) ...
    # Result shape: (N, 136) where 136 = 17 * 4 * 2
    output_flat = output_state.reshape(output_state.shape[0], -1)

    # 2. NORMALIZE (Global Scalar approach as per your previous code)
    mean = np.mean(output_flat, axis=0)
    std = np.std(output_flat) # Global standard deviation
    
    norm_output = (output_flat - mean) / std

    # 3. COMBINE with Period
    normalized_targets = np.hstack([norm_output, period_data])
    
    return normalized_targets, mean, std

def denormalize(prediction_flat, mean, std):
    """
    Args:
        prediction_flat: The predicted frequency part (N, 136) or (136,)
        mean: The saved mean vector (136,)
        std: The saved global std scalar
    Returns:
        The denormalized flat data
    """
    return (prediction_flat * std) + mean

def recover_shape(flat_data):
    """
    Args:
        flat_data: Numpy array of shape (136,) 
                   Contains flattened FFT coefficients.
    Returns:
        structured_data: Shape (4, 2, 17) -> (Joints, Real/Imag, Freqs)
                         This is ready for your 'pred_ifft' function.
    """
    # 1. RESHAPE to original creation shape: (Freqs, Joints, Real/Imag)
    # This must match the dimensions of 'joint_states_rfft' from extract_gait_cycle
    # shape becomes (17, 4, 2)
    recovered = flat_data.reshape(17, 4, 2)
    
    # 2. TRANSPOSE to the shape expected by your plotting/ifft function
    # Current indices: 0=Freq, 1=Joint, 2=Real/Imag
    # Target indices:  1=Joint, 2=Real/Imag, 0=Freq
    structured_data = recovered.transpose(1, 2, 0)
    
    return structured_data

# Usage Example during dataset creation:
# total_output_state shape is (N, 17, 4, 2)
targets, mean, std = normalize_and_flatten(total_output_state, total_period)

# Save these for training/inference
np.save("gait reference phase 2/mean.npy", mean)
np.save("gait reference phase 2/std.npy", std)
np.save("gait reference phase 2/targets.npy", targets)

**Inverse FFT**

In [55]:
def pred_ifft(predictions,ground_truth,speed):
    #form is [4,2,17]
    real_pred = predictions[:,0,:]
    imag_pred = predictions[:,1,:]
    predictions = real_pred + 1j*imag_pred
    real_gt = ground_truth[:,0,:]
    imag_gt = ground_truth[:,1,:]
    ground_truth = real_gt + 1j*imag_gt
    

    pred_time = np.fft.irfft(predictions, axis=1)
    gt_time = np.fft.irfft(ground_truth, axis=1)
    pred_time = pred_time.transpose(1,0)
    gt_time = gt_time.transpose(1,0)

    return pred_time,gt_time

**Define Architechture**

In [56]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Create the results folder if it does not exist.
# -----------------------------

results_dir = "ref_gait_results"
os.makedirs(results_dir, exist_ok=True)

class SimpleFCNN(nn.Module):
    def __init__(self, input_size=3, output_size=204, hidden_size=512):
        super(SimpleFCNN, self).__init__()
        # Increased depth slightly to help map the non-linear relationship at low speeds
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LeakyReLU(0.1),  # LeakyReLU often trains better for signal regression
            nn.Linear(hidden_size, hidden_size),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_size, hidden_size), # Added one more layer for capacity
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_size, output_size)
        )
        
    def forward(self, x):
        return self.net(x)


**K-Fold Validation for Better Training**

In [57]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
import itertools

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
DATA_DIR = "gait reference phase 2"
RESULTS_ROOT = "kfold_results"

# Load Data
try:
    inputs = np.load(os.path.join(DATA_DIR, "input_vector.npy"))
    outputs = np.load(os.path.join(DATA_DIR, "targets.npy"))
    mean = np.load(os.path.join(DATA_DIR, "mean.npy")) # Ensure you saved these in prev step
    std = np.load(os.path.join(DATA_DIR, "std.npy"))   # Ensure you saved these in prev step
except FileNotFoundError:
    print("Error: specific data files not found. Please check paths.")
    # Dummy data for code validation if files missing
    inputs = np.random.rand(100, 3).astype(np.float32)
    outputs = np.random.rand(100, 137).astype(np.float32)
    mean = np.zeros(136)
    std = 1.0

# Convert to Float32
inputs = inputs.astype(np.float32)
outputs = outputs.astype(np.float32)

print(f"Data Loaded. Inputs: {inputs.shape}, Outputs: {outputs.shape}")

# ==========================================
# 2. HELPER FUNCTIONS (CORE LOGIC)
# ==========================================

class SimpleFCNN(nn.Module):
    def __init__(self, input_size=3, output_size=137, hidden_size=512):
        super(SimpleFCNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_size, hidden_size),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_size, hidden_size),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_size, output_size)
        )
        
    def forward(self, x):
        return self.net(x)

def denormalize(pred_flat, gt_flat, mean, std):
    """Denormalizes flattening frequency data."""
    pred = (pred_flat * std) + mean
    gt = (gt_flat * std) + mean
    return pred, gt

def recover_shape(flat_data):
    """
    FIXED RESHAPE FUNCTION
    Reconstructs (4, 2, 17) from flat (136,) vector.
    """
    # 1. Reshape to creation shape: (Freqs=17, Joints=4, Real/Imag=2)
    recovered = flat_data.reshape(17, 4, 2)
    # 2. Transpose to IFFT shape: (Joints=4, Real/Imag=2, Freqs=17)
    structured = recovered.transpose(1, 2, 0)
    return structured

def pred_ifft(predictions, ground_truth):
    """
    Performs Inverse FFT to get time-domain signals.
    Input Shape: (4, 2, 17) -> [Joints, Real/Imag, Freqs]
    """
    # Combine Real and Imaginary parts
    # predictions[:, 0, :] is Real, predictions[:, 1, :] is Imag
    complex_pred = predictions[:, 0, :] + 1j * predictions[:, 1, :]
    complex_gt   = ground_truth[:, 0, :] + 1j * ground_truth[:, 1, :]
    
    # Inverse FFT (n=32 points)
    pred_time = np.fft.irfft(complex_pred, n=32, axis=1)
    gt_time   = np.fft.irfft(complex_gt, n=32, axis=1)
    
    # Transpose for plotting: (Time, Joints)
    return pred_time.T, gt_time.T

# ==========================================
# 3. HYPERPARAMETERS GRID
# ==========================================
# Redefined priority hyperparameters
param_grid = {
    'batch_size': [32],           # Reduced for demonstration, add [64] if needed
    'hidden_size': [256, 512],
    'learning_rate': [1e-3, 3e-4],
    'num_epochs': [2000]
}

# Generate all combinations
keys, values = zip(*param_grid.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

# Constants
K_FOLDS = 10
INPUT_SIZE = 3
OUTPUT_SIZE = 137
FREQ_DIM = 136
PERIOD_WEIGHT = 5.0

# ==========================================
# 4. K-FOLD VALIDATION LOOP
# ==========================================

best_global_loss = float('inf')
best_global_config = None

print(f"Starting Grid Search with {len(experiments)} configurations...")

for i, config in enumerate(experiments):
    bs = config['batch_size']
    hs = config['hidden_size']
    lr = config['learning_rate']
    epochs = config['num_epochs']
    
    config_name = f"BS{bs}_HS{hs}_LR{lr}_EP{epochs}"
    config_dir = os.path.join(RESULTS_ROOT, config_name)
    os.makedirs(config_dir, exist_ok=True)
    
    print(f"\n--- Running Config {i+1}/{len(experiments)}: {config_name} ---")
    
    kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    fold_val_losses = []
    
    # --- K-FOLD LOOP ---
    for fold, (train_ids, val_ids) in enumerate(kfold.split(inputs, outputs)):
        print(f"  Fold {fold+1}/{K_FOLDS}...", end="\r")
        
        # Create Fold Directory
        fold_dir = os.path.join(config_dir, f"fold_{fold+1}")
        os.makedirs(fold_dir, exist_ok=True)
        plot_dir = os.path.join(fold_dir, "plots")
        os.makedirs(plot_dir, exist_ok=True)
        
        # Prepare Data
        X_train_fold = torch.tensor(inputs[train_ids], dtype=torch.float32)
        y_train_fold = torch.tensor(outputs[train_ids], dtype=torch.float32)
        X_val_fold   = torch.tensor(inputs[val_ids], dtype=torch.float32)
        y_val_fold   = torch.tensor(outputs[val_ids], dtype=torch.float32)
        
        train_loader = DataLoader(TensorDataset(X_train_fold, y_train_fold), batch_size=bs, shuffle=True)
        val_loader   = DataLoader(TensorDataset(X_val_fold, y_val_fold), batch_size=1, shuffle=False)
        
        # Initialize Model
        model = SimpleFCNN(input_size=INPUT_SIZE, output_size=OUTPUT_SIZE, hidden_size=hs)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.MSELoss()
        
        # Training
        best_fold_loss = float('inf')
        
        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            for x, y in train_loader:
                optimizer.zero_grad()
                pred = model(x)
                
                # Split loss
                loss_freq = loss_fn(pred[:, :FREQ_DIM], y[:, :FREQ_DIM])
                loss_period = loss_fn(pred[:, FREQ_DIM:], y[:, FREQ_DIM:])
                loss = loss_freq + (PERIOD_WEIGHT * loss_period)
                
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * x.size(0)
            
            # Validation (Simple check for model saving)
            # We save the model at the very end or track best val loss here
            # For speed, we just train and evaluate at end, or track best val
        
        # Final Evaluation of this Fold
        model.eval()
        val_loss_accum = 0.0
        
        with torch.no_grad():
            for idx, (x_val, y_val) in enumerate(val_loader):
                # Forward
                pred_out = model(x_val)
                
                # Loss
                l_freq = loss_fn(pred_out[:, :FREQ_DIM], y_val[:, :FREQ_DIM])
                l_period = loss_fn(pred_out[:, FREQ_DIM:], y_val[:, FREQ_DIM:])
                total_l = l_freq + (PERIOD_WEIGHT * l_period)
                val_loss_accum += total_l.item()
                
                # Plotting (Save plots for first 5 samples of the fold)
                if idx < 5: 
                    # Extract Data
                    speed = x_val[0, 0].item() * 2.4 # Assuming scaling
                    
                    freqs_pred = pred_out[0, :FREQ_DIM].numpy()
                    freqs_gt   = y_val[0, :FREQ_DIM].numpy()
                    per_pred   = pred_out[0, FREQ_DIM].item()
                    per_gt     = y_val[0, FREQ_DIM].item()
                    
                    # 1. Denormalize
                    f_pred_dn, f_gt_dn = denormalize(freqs_pred, freqs_gt, mean, std)
                    
                    # 2. Recover Shape
                    struct_pred = recover_shape(f_pred_dn)
                    struct_gt   = recover_shape(f_gt_dn)
                    
                    # 3. IFFT
                    pred_t, gt_t = pred_ifft(struct_pred, struct_gt)
                    
                    # 4. Plot
                    fig, axs = plt.subplots(2, 2, figsize=(10, 8))
                    channels = ["R Hip", "R Knee", "L Hip", "L Knee"]
                    for c_idx, ax in enumerate(axs.flat):
                        ax.plot(gt_t[:, c_idx], label="GT", color='tab:blue')
                        ax.plot(pred_t[:, c_idx], label="Pred", color='tab:orange', linestyle='--')
                        ax.set_title(channels[c_idx])
                        ax.grid(True)
                    
                    fig.suptitle(f"Speed: {speed:.2f} | Period: GT {per_gt:.2f} vs Pred {per_pred:.2f}")
                    plt.tight_layout()
                    plt.savefig(os.path.join(plot_dir, f"sample_{idx}.png"))
                    plt.close()

        avg_fold_loss = val_loss_accum / len(val_loader)
        fold_val_losses.append(avg_fold_loss)
        
        # Save Fold Model
        torch.save(model.state_dict(), os.path.join(fold_dir, "model.pth"))
    
    # Calculate Average Loss across all folds for this Config
    avg_config_loss = np.mean(fold_val_losses)
    print(f"\n  > Config Finished. Avg K-Fold Loss: {avg_config_loss:.6f}")
    
    # Track Best Global
    if avg_config_loss < best_global_loss:
        best_global_loss = avg_config_loss
        best_global_config = config
        print(f"  *** NEW BEST CONFIG FOUND ***")

# ==========================================
# 5. FINAL FULL TRAINING
# ==========================================
print("\n" + "="*40)
print(f"GRID SEARCH COMPLETE.")
print(f"Best Config: {best_global_config}")
print(f"Best Avg Loss: {best_global_loss:.6f}")
print("="*40)
print("Starting Final Training on Full Dataset...")

# Use Best Hyperparams
final_bs = best_global_config['batch_size']
final_hs = best_global_config['hidden_size']
final_lr = best_global_config['learning_rate']
final_ep = best_global_config['num_epochs']

# Full Dataset Loader
full_dataset = TensorDataset(torch.tensor(inputs), torch.tensor(outputs))
full_loader = DataLoader(full_dataset, batch_size=final_bs, shuffle=True)

# Final Model
final_model = SimpleFCNN(input_size=INPUT_SIZE, output_size=OUTPUT_SIZE, hidden_size=final_hs)
optimizer = optim.Adam(final_model.parameters(), lr=final_lr)
loss_fn = nn.MSELoss()

# Train Loop
loss_history = []
for epoch in range(final_ep):
    final_model.train()
    epoch_loss = 0.0
    for x, y in full_loader:
        optimizer.zero_grad()
        pred = final_model(x)
        
        l_freq = loss_fn(pred[:, :FREQ_DIM], y[:, :FREQ_DIM])
        l_period = loss_fn(pred[:, FREQ_DIM:], y[:, FREQ_DIM:])
        loss = l_freq + (PERIOD_WEIGHT * l_period)
        
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x.size(0)
        
    avg_loss = epoch_loss / len(full_dataset)
    loss_history.append(avg_loss)
    
    if (epoch+1) % 100 == 0:
        print(f"Final Train Epoch {epoch+1}/{final_ep} | Loss: {avg_loss:.6f}")

# Save Final Model
final_path = os.path.join(RESULTS_ROOT, "FINAL_BEST_MODEL.pth")
torch.save(final_model.state_dict(), final_path)
print(f"Final Model Saved to: {final_path}")

# Plot Training Curve
plt.figure()
plt.plot(loss_history)
plt.title("Final Model Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.savefig(os.path.join(RESULTS_ROOT, "final_training_loss.png"))
plt.close()

Data Loaded. Inputs: (1121, 3), Outputs: (1121, 137)
Starting Grid Search with 4 configurations...

--- Running Config 1/4: BS32_HS256_LR0.001_EP2000 ---
  Fold 10/10...
  > Config Finished. Avg K-Fold Loss: 0.092021
  *** NEW BEST CONFIG FOUND ***

--- Running Config 2/4: BS32_HS256_LR0.0003_EP2000 ---
  Fold 10/10...
  > Config Finished. Avg K-Fold Loss: 0.099905

--- Running Config 3/4: BS32_HS512_LR0.001_EP2000 ---
  Fold 10/10...
  > Config Finished. Avg K-Fold Loss: 0.092322

--- Running Config 4/4: BS32_HS512_LR0.0003_EP2000 ---
  Fold 10/10...
  > Config Finished. Avg K-Fold Loss: 0.089290
  *** NEW BEST CONFIG FOUND ***

GRID SEARCH COMPLETE.
Best Config: {'batch_size': 32, 'hidden_size': 512, 'learning_rate': 0.0003, 'num_epochs': 2000}
Best Avg Loss: 0.089290
Starting Final Training on Full Dataset...
Final Train Epoch 100/2000 | Loss: 0.133441
Final Train Epoch 200/2000 | Loss: 0.123450
Final Train Epoch 300/2000 | Loss: 0.115519
Final Train Epoch 400/2000 | Loss: 0.111380


**Training Dataset Creation and Script ##Currently Commented Out**

In [ ]:
# import scipy
# import numpy as np
# import matplotlib.pyplot as plt
# import torch
# from torch import nn
# import torch.optim as optim
# import torch.nn.functional as F
# from torch.utils.data import DataLoader, TensorDataset

# inputs = np.load(rf"gait reference phase 2/input_vector.npy")
# outputs = np.load(rf"gait reference phase 2\targets.npy")
# # period = np.load(rf"gait reference phase 2/period.npy")
# #train test split using sklearn
# # outputs = outputs.transpose(0,2,3,1)
# # outputs = outputs.reshape(outputs.shape[0],-1)
# print(outputs.shape)

# from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(inputs, outputs, test_size=0.15, random_state=23)

# #convert to tensor
# X_train = torch.tensor(X_train, dtype=torch.float32)
# y_train = torch.tensor(y_train, dtype=torch.float32)

# X_test = torch.tensor(X_test, dtype=torch.float32)
# y_test = torch.tensor(y_test, dtype=torch.float32)

# # create dataloader
# train_data = TensorDataset(X_train, y_train)
# val_data = TensorDataset(X_test, y_test)

# print('train data shape: ',X_train.shape)
# print('test data shape: ',X_test.shape)
# print('train output shape: ',y_train.shape)
# print('test output shape: ',y_test.shape)

# # -----------------------------
# # Hyperparameters to tune
# # -----------------------------

# batch_sizes = [32, 64]
# hidden_sizes = [256, 512]
# num_epochs_list = [2000]
# learning_rates = [1e-3, 3e-4]

# input_size = 3
# output_size = 137

# # Split dims: first 136 = freq, last 1 = period
# freq_dim = output_size - 1
# period_weight = 5.0

# torch.manual_seed(23)

# tuning_results = []

# best_params = None
# best_model_state = None

# os.makedirs(results_dir, exist_ok=True)

# for bs in batch_sizes:
#     train_loader = DataLoader(train_data, batch_size=bs, shuffle=True)
#     val_loader = DataLoader(val_data, batch_size=bs, shuffle=False)

#     for hs in hidden_sizes:
#         for num_epochs in num_epochs_list:
#             for lr in learning_rates:
#                 print(f"Training with batch_size={bs}, hidden_size={hs}, epochs={num_epochs}, lr={lr}")

#                 torch.manual_seed(42)  # reproducibility per run
#                 model = SimpleFCNN(input_size=input_size, output_size=output_size, hidden_size=hs)
#                 optimizer = optim.Adam(model.parameters(), lr=lr)
#                 loss_fn = nn.MSELoss()

#                 train_losses = []
#                 val_losses = []
#                 saved_epoch = None
#                 best_val_loss = float("inf")
#                 for epoch in range(num_epochs):
#                     # -----------------
#                     # Train
#                     # -----------------
#                     model.train()
#                     running_train_loss = 0.0

#                     for inputs, targets in train_loader:
#                         targets = targets.view(-1, output_size)

#                         optimizer.zero_grad()
#                         outputs = model(inputs)

#                         out_freq = outputs[:, :freq_dim]
#                         out_period = outputs[:, freq_dim:]  # [B,1]
#                         tgt_freq = targets[:, :freq_dim]
#                         tgt_period = targets[:, freq_dim:]  # [B,1]

#                         loss_freq = loss_fn(out_freq, tgt_freq)
#                         loss_period = loss_fn(out_period, tgt_period)

#                         loss = loss_freq + period_weight * loss_period

#                         loss.backward()
#                         optimizer.step()

#                         running_train_loss += loss.item() * inputs.size(0)

#                     epoch_train_loss = running_train_loss / len(train_loader.dataset)
#                     train_losses.append(epoch_train_loss)

#                     # -----------------
#                     # Validate (EVERY epoch)
#                     # -----------------
#                     model.eval()
#                     running_val_loss = 0.0
#                     with torch.no_grad():
#                         for inputs, targets in val_loader:
#                             targets = targets.view(-1, output_size)
#                             outputs = model(inputs)

#                             out_freq = outputs[:, :freq_dim]
#                             out_period = outputs[:, freq_dim:]
#                             tgt_freq = targets[:, :freq_dim]
#                             tgt_period = targets[:, freq_dim:]

#                             loss_freq = loss_fn(out_freq, tgt_freq)
#                             loss_period = loss_fn(out_period, tgt_period)

#                             loss = loss_freq + period_weight * loss_period
#                             running_val_loss += loss.item() * inputs.size(0)

#                     epoch_val_loss = running_val_loss / len(val_loader.dataset)
#                     val_losses.append(epoch_val_loss)

#                     # -----------------
#                     # Logging + tracking best
#                     # -----------------
#                     if (epoch + 1) % 10 == 0 or epoch == num_epochs - 1:
#                         print(
#                             f"  Epoch {epoch+1}/{num_epochs} | "
#                             f"Train Loss: {epoch_train_loss:.6f} | Val Loss: {epoch_val_loss:.6f}"
#                         )

#                     # Save "best" based on current epoch val (not sparse val list)
#                     if epoch_val_loss < best_val_loss:
#                         saved_epoch = epoch + 1
#                         best_val_loss = epoch_val_loss
#                         best_params = {
#                             "batch_size": bs,
#                             "hidden_size": hs,
#                             "num_epochs": epoch + 1,
#                             "learning_rate": lr,
#                             "final_train_loss": epoch_train_loss,
#                             "final_val_loss": epoch_val_loss,
#                         }
#                         best_model_state = model.state_dict()

#                         model_name = (
#                             f"phase_2_best_model_hs{hs}_lr{lr}_bs{bs}.pth"
#                         )
#                         torch.save(best_model_state, os.path.join(results_dir, model_name))
#                         print(f"  New best model saved: {model_name}")

#                 # Record final results from this run (end of training)
#                 tuning_results.append(
#                     {
#                         "batch_size": bs,
#                         "hidden_size": hs,
#                         "num_epochs": num_epochs,
#                         "learning_rate": lr,
#                         "final_train_loss": train_losses[-1],
#                         "final_val_loss": val_losses[-1],
#                     }
#                 )

#                 # -----------------
#                 # Save loss plot
#                 # -----------------
#                 plt.figure()
#                 plt.plot(np.arange(1, num_epochs + 1), train_losses, label="Train Loss")
#                 plt.plot(np.arange(1, num_epochs + 1), val_losses, label="Val Loss")
#                 plt.axvline(x=saved_epoch, color='r', linestyle='--', label='Saved Epoch')
#                 plt.xlabel("Epoch")
#                 plt.ylabel("Loss")
#                 plt.legend()
#                 plt.title(f"bs={bs}, hs={hs}, epochs={num_epochs}, lr={lr}, Val Loss: {best_val_loss:.4f}")
#                 plt.tight_layout()

#                 plot_filename = f"phase_2_loss_plot_bs{bs}_hs{hs}_epochs{num_epochs}_lr{lr}.png"
#                 plt.savefig(os.path.join(results_dir, plot_filename))
#                 plt.close()
#                 print(f"Loss plot saved as {os.path.join(results_dir, plot_filename)}\n")

# print("Best params:", best_params)
# print("Best val loss:", best_val_loss)


In [ ]:
def pred_ifft_test_plot(predictions,ground_truth,speed,period_gt,period_pred):
    #form is [5,2,17]
    real_pred = predictions[:,:,0]
    imag_pred = predictions[:,:,1]
    predictions = real_pred + 1j*imag_pred
    real_gt = ground_truth[:,:,0]
    imag_gt = ground_truth[:,:,1]
    ground_truth = real_gt + 1j*imag_gt
    

    pred_time = np.fft.irfft(predictions, axis=1)
    gt_time = np.fft.irfft(ground_truth, axis=1)
    pred_time = pred_time.transpose(1,0)
    gt_time = gt_time.transpose(1,0)
    #plot 2*2 subplots
    # plt.figure(figsize=(10,8))
    # plt.suptitle(f"period_gt = {period_gt:.2f}, period = {period_pred:.2f}")

    # plt.subplot(2,2,1)
    # plt.plot(pred_time[:,0])
    # plt.plot(gt_time[:,0])
    # plt.title('Right Hip')
    # plt.legend(['Predicted','Ground Truth'])
    # plt.subplot(2,2,2)
    # plt.plot(pred_time[:,1])
    # plt.plot(gt_time[:,1])
    # plt.title('Right Knee')
    # plt.legend(['Predicted','Ground Truth'])

    # plt.subplot(2,2,3)
    # plt.plot(pred_time[:,2])
    # plt.plot(gt_time[:,2])
    # plt.title('Left Hip')
    # plt.legend(['Predicted','Ground Truth'])
    # plt.subplot(2,2,4)
    # plt.plot(pred_time[:,3])
    # plt.plot(gt_time[:,3])
    # plt.title('Left Knee')
    # plt.legend(['Predicted','Ground Truth'])


    # plt.savefig(f"compare_phase2/bs64_{speed:.2f}ms.png")
    # plt.close()
    # print('plots saved')
    return pred_time,gt_time

In [ ]:
# test_dataloader = DataLoader(val_data, batch_size=1, shuffle=False)  # Batch size of 5
# # Testing loop
# model = SimpleFCNN(input_size=3, output_size=137, hidden_size=512)
# model_name = r"C:\Users\yusuf\Bipedal-imitation-rl\ref_gait_results\phase_2_best_model_hs512_lr0.0003_bs32.pth"
# model.load_state_dict(torch.load(model_name))
# model.eval()  # Set the model to evaluation mode
# correct = 0
# total = 0
# test_loss = 0.0

# loss_fn = nn.MSELoss()

# k = 0
# mean = np.load(rf"C:\Users\yusuf\Bipedal-imitation-rl\gait reference phase 2\fft_mean.npy")
# std = np.load(rf"C:\Users\yusuf\Bipedal-imitation-rl\gait reference phase 2\fft_global_std.npy")

# with torch.no_grad():  # No need to compute gradients during testing
#     for inputs, targets in test_dataloader:
#         speed = inputs[0,0].item()*2.4
#         # Forward pass

#         outputs = model(inputs)

#         # Remove fake dimension if present (protect if batch_empty)
#         # outputs: [1, 137], targets: [1, 137] for batch_size=1
#         outputs = outputs.squeeze(0)
#         targets = targets.squeeze(0)

#         freqs = outputs[:-1]
#         period_pred = outputs[-1]
#         freqs_gt = targets[:-1]
#         period_gt = targets[-1]

#         # Compute loss
#         loss = loss_fn(outputs, targets)
#         test_loss += loss.item()

#         predictions, ground_truth = denormalize(freqs,freqs_gt,mean,std)

#         ground_truth = ground_truth.reshape(17,4,2)
#         predictions = predictions.reshape(17,4,2)

#         pred_time,gt_time = pred_ifft_test_plot(predictions,ground_truth,speed,period_gt,period_pred)
        
#         k+=1
#         fig, axs = plt.subplots(2, 2, figsize=(10, 6))
#         fig.suptitle(f"speed={speed:.2f} Period_gt : {period_gt:.2f} Period_pred : {period_pred:.2f}")

#         # Plot channel 0
#         axs[0, 0].plot(gt_time[:, 0], label='GT')
#         axs[0, 0].plot(pred_time[:, 0], linestyle='--', label='Pred')
#         axs[0, 0].set_title('Channel 0')
#         axs[0, 0].legend()
#         axs[0, 0].grid(True)

#         # Plot channel 1
#         axs[0, 1].plot(gt_time[:, 1], label='GT')
#         axs[0, 1].plot(pred_time[:, 1], linestyle='--', label='Pred')
#         axs[0, 1].set_title('Channel 1')
#         axs[0, 1].legend()
#         axs[0, 1].grid(True)

#         # Plot channel 2
#         axs[1, 0].plot(gt_time[:, 2], label='GT')
#         axs[1, 0].plot(pred_time[:, 2], linestyle='--', label='Pred')
#         axs[1, 0].set_title('Channel 2')
#         axs[1, 0].legend()
#         axs[1, 0].grid(True)

#         # Plot channel 3
#         axs[1, 1].plot(gt_time[:, 3], label='GT')
#         axs[1, 1].plot(pred_time[:, 3], linestyle='--', label='Pred')
#         axs[1, 1].set_title('Channel 3')
#         axs[1, 1].legend()
#         axs[1, 1].grid(True)

#         plt.tight_layout()
#         plt.show()
# # Calculate average loss and accuracy
# test_loss /= len(test_dataloader)  # Average test loss


# # Print test results
# print(f"Test Loss: {test_loss:.4f}")
# plt.plot(targets[0].numpy())
# plt.plot(outputs[0].numpy())
# plt.legend(['Ground Truth', 'Predictions'])

**Train the Network with the Full Data**

In [ ]:
# inputs = np.load(rf"gait reference fft5.00/newnormalized_input_vector.npy")
# outputs = np.load(rf"gait reference fft5.00/newnormalized_output_fft_constants.npy")

# outputs = outputs.transpose(0,2,3,1)
# outputs = torch.tensor(outputs, dtype=torch.float32)
# outputs = outputs.reshape(outputs.shape[0],-1)
# inputs = torch.tensor(inputs, dtype=torch.float32)
# train_data = TensorDataset(inputs, outputs)

# # ADJUST THE PARAMETERS BASED ON THE BEST VALIDATION SCORED RESULT
# bs = best_bs
# hs = best_hs
# lr = best_lr
# train_loader = DataLoader(train_data, batch_size=bs, shuffle=True)

# num_epochs = best_epoch
# torch.manual_seed(23)  # Ensure reproducibility per run.
# model = SimpleFCNN(input_size=input_size, output_size=output_size, hidden_size=hs)
# optimizer = optim.Adam(model.parameters(), lr=lr)
# loss_fn = nn.MSELoss()

# train_losses = []
# val_losses = []

# # Training loop for the current hyperparameter combination.
# for epoch in range(num_epochs):
#     model.train()
#     running_train_loss = 0.0
#     for inputs, targets in train_loader:
#         # Ensure targets are the right shape.
#         targets = targets.view(-1, output_size)
#         optimizer.zero_grad()
#         outputs = model(inputs)
#         loss = loss_fn(outputs, targets)
#         loss.backward()
#         optimizer.step()
#         running_train_loss += loss.item() * inputs.size(0)
#     epoch_train_loss = running_train_loss / len(train_loader.dataset)
#     train_losses.append(epoch_train_loss)
#     if (epoch+1) % 200 == 0 or epoch == num_epochs - 1:
#         print(f"  Epoch {epoch+1}/{num_epochs} | Train Loss: {epoch_train_loss:.4f}")

# model_filename = f"final_model.pth"
# torch.save(model.state_dict(), os.path.join(model_filename))